> **Research record.** Cleaned copy of `500-research-home-work/102-identify-home-change-migration-panel.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Builds the user x work-location x month panel that links remote-work intensity to home relocation.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["GRPC_DNS_RESOLVER"] = "native"

from pyproj import datadir

env_proj = os.path.join(sys.prefix, "share", "proj")
if os.path.isdir(env_proj):
    os.environ["PROJ_DATA"] = env_proj
    os.environ["PROJ_LIB"] = env_proj
    datadir.set_data_dir(env_proj)

prepare_data_path = Path().absolute().parent
if str(prepare_data_path) not in sys.path:
    sys.path.insert(0, str(prepare_data_path))

In [ ]:
from geofunctions import utils
import pandas as pd

In [ ]:
df_users = utils.read_query_to_dataframe(
    """select c.*,
       SAs as yishuv_stat, shem_yishuv_heb, 
       case when SAsType =1 then 'CBD' when SAsType =2 then 'residential_central' when SAsType =3 then 'residential_remote' when SAsType =4 then 'satellite_city' end as new_neighborhood_type,
       distance_to_azrieli,
       flag_remote_work_smoothed_max_thresh
       from `phd-habidatum.tmp.identifiers_with_home_loc_changes` c
       join phd-habidatum.phd_adm.geohashes_with_sa_gushdan gs
       on geohash_home=gs.geohash
       join phd-habidatum.phd_adm.match_census_2022_2011 m
       on cast(yeshuv_stat_2022 as string) = cast(gs.yishuv_stat_2022 as string)
       join phd-habidatum.phd_datasets.panel_types_matan s
       on yishuv_stat11 = cast(SAs as string)
       join phd-habidatum.phd_datasets.sa_neigborhoods_types s2
       using (yishuv_stat_2022)
       join phd-habidatum.phd_users_locations.users_remote_work_monthly urw
       using (identifier, month)
       where total_work_days>=5 -- to reduce noise
       """
)

In [ ]:
def create_remote_work_home_change_table(
    df_users,
    user_col="identifier",
    work_col="geohash_work",
    month_col="month",
    home_col="geohash_home",
    neighborhood_col="new_neighborhood_type",
    neighborhood_id_col="yishuv_stat",
    distance_office_home_col="distance_office_home",
    distance_to_azrieli_col="distance_to_azrieli",
    remote_work_col="flag_remote_work_smoothed_max_thresh",
    start_month="2020-01-01",
    lookback_months=3,
    horizon_months=6,
):
    """
    For each (identifier, geohash_work) unit-month from start_month, calculate trailing
    average remote work, forward home-change outcomes, and forward remote-work averages.

    Calendar windows are implemented as integer-month offset joins (no per-unit Python loop).
    """
    unit_key = [user_col, work_col]
    df = df_users.copy()
    df[month_col] = pd.to_datetime(df[month_col])
    start_month = pd.Timestamp(start_month)

    df_panel = (
        df.dropna(
            subset=unit_key
            + [month_col, home_col, neighborhood_col, neighborhood_id_col]
        )
        .sort_values(unit_key + [month_col])
        .drop_duplicates(unit_key + [month_col], keep="last")[
            unit_key
            + [
                month_col,
                home_col,
                neighborhood_col,
                neighborhood_id_col,
                distance_office_home_col,
                distance_to_azrieli_col,
                remote_work_col,
            ]
        ]
        .copy()
    )
    df_panel["neighborhood_id"] = df_panel[neighborhood_id_col].astype(str)

    # Lookup: geohash_home → home neighborhood type (not work neighborhood)
    home_to_neighborhood = (
        df.dropna(subset=[home_col, neighborhood_col])
        .drop_duplicates(home_col)
        .set_index(home_col)[neighborhood_col]
    )
    home_to_distance = (
        df.dropna(subset=[home_col, distance_to_azrieli_col])
        .drop_duplicates(home_col)
        .set_index(home_col)[distance_to_azrieli_col]
    )
    df_panel["home_neighborhood_type"] = df_panel[home_col].map(home_to_neighborhood)
    df_panel["home_distance_to_azrieli"] = df_panel[home_col].map(home_to_distance)
    df_panel[remote_work_col] = df_panel[remote_work_col].fillna(0)
    df_panel = df_panel[df_panel[month_col] >= start_month].copy()

    df_panel = df_panel.reset_index(drop=True)
    df_panel["_ym"] = df_panel[month_col].dt.year * 12 + df_panel[month_col].dt.month
    origin_keys = unit_key + [month_col]
    change_col = f"flag_home_location_changed_next_{horizon_months}m"
    n_col = f"n_months_with_home_next_{horizon_months}m"

    series = df_panel[
        unit_key
        + [
            "_ym",
            home_col,
            remote_work_col,
            "home_neighborhood_type",
            "home_distance_to_azrieli",
            neighborhood_id_col,
            distance_office_home_col,
        ]
    ].rename(
        columns={
            "_ym": "_ym_f",
            home_col: "home_f",
            remote_work_col: "rw_f",
            "home_neighborhood_type": "nbhd_f",
            "home_distance_to_azrieli": "az_f",
            neighborhood_id_col: "yishuv_f",
            distance_office_home_col: "office_home_f",
        }
    )

    def join_offsets(left, ym_col, offsets, extra=()):
        cols = list(dict.fromkeys(origin_keys + [ym_col] + list(extra)))
        base = left[cols]
        frames = []
        for k in offsets:
            tmp = base.copy()
            tmp["_k"] = k
            tmp["_ym_f"] = tmp[ym_col] + k
            frames.append(tmp)
        return pd.concat(frames, ignore_index=True).merge(
            series, on=unit_key + ["_ym_f"], how="left"
        )

    lookback = join_offsets(
        df_panel, "_ym", range(-(lookback_months - 1), 1)
    ).dropna(subset=["rw_f"])
    avg_rw = lookback.groupby(origin_keys, as_index=False).agg(
        avg_flag_remote_work=("rw_f", "mean")
    )

    fut06 = join_offsets(
        df_panel, "_ym", range(1, horizon_months + 1), extra=(home_col,)
    )
    fut06_hit = fut06.dropna(subset=["home_f"]).copy()
    fut06_hit["_changed"] = fut06_hit["home_f"] != fut06_hit[home_col]
    fut06_stats = fut06_hit.groupby(origin_keys, as_index=False).agg(
        **{
            n_col: ("home_f", "count"),
            "avg_flag_remote_work_followup_0_6m": ("rw_f", "mean"),
            change_col: ("_changed", "any"),
        }
    )

    first_src = fut06_hit.loc[fut06_hit["_changed"]].sort_values(
        origin_keys + ["_k"]
    )
    first = first_src.groupby(origin_keys, as_index=False).first()
    first = first.rename(
        columns={
            "home_f": "h1",
            "_ym_f": "_ym_tp",
            "nbhd_f": "new_home_neighborhood_type",
            "az_f": "new_home_distance_to_azrieli",
            "yishuv_f": "new_home_yishuv_stat",
            "office_home_f": "new_home_distance_office_home",
        }
    )
    first_keep = origin_keys + [
        home_col,
        "h1",
        "_ym_tp",
        "new_home_neighborhood_type",
        "new_home_distance_to_azrieli",
        "new_home_yishuv_stat",
        "new_home_distance_office_home",
    ]
    first = first.reindex(columns=first_keep)

    if len(first):
        after = fut06_hit.merge(
            first[origin_keys + ["_ym_tp"]], on=origin_keys, how="inner"
        )
        after = after[after["_ym_f"] >= after["_ym_tp"]]
        rw_after = after.groupby(origin_keys, as_index=False).agg(
            avg_flag_remote_work_after_new_home=("rw_f", "mean")
        )
        pers = join_offsets(
            first, "_ym_tp", range(0, 7), extra=("h1", home_col)
        ).dropna(subset=["home_f"])
        pers = pers.copy()
        pers["is_new"] = pers["home_f"] == pers["h1"]
        pers["is_orig"] = pers["home_f"] == pers[home_col]
        pers["is_third"] = ~pers["is_new"] & ~pers["is_orig"]
        pers_counts = pers.groupby(origin_keys, as_index=False).agg(
            n_months_new_home_t1_to_tp6=("is_new", "sum"),
            n_months_original_home_t1_to_tp6=("is_orig", "sum"),
            n_months_third_home_t1_to_tp6=("is_third", "sum"),
        )
    else:
        rw_after = pd.DataFrame(
            columns=origin_keys + ["avg_flag_remote_work_after_new_home"]
        )
        pers_counts = pd.DataFrame(
            columns=origin_keys
            + [
                "n_months_new_home_t1_to_tp6",
                "n_months_original_home_t1_to_tp6",
                "n_months_third_home_t1_to_tp6",
            ]
        )

    fut69 = join_offsets(
        df_panel, "_ym", range(horizon_months + 1, 10)
    ).dropna(subset=["rw_f"])
    rw69 = fut69.groupby(origin_keys, as_index=False).agg(
        avg_flag_remote_work_followup_6_9m=("rw_f", "mean")
    )

    df_panel = (
        df_panel.merge(avg_rw, on=origin_keys, how="left")
        .merge(fut06_stats, on=origin_keys, how="left")
        .merge(rw69, on=origin_keys, how="left")
        .merge(
            first[
                origin_keys
                + [
                    "new_home_neighborhood_type",
                    "new_home_distance_to_azrieli",
                    "new_home_yishuv_stat",
                    "new_home_distance_office_home",
                ]
            ],
            on=origin_keys,
            how="left",
        )
        .merge(rw_after, on=origin_keys, how="left")
        .merge(pers_counts, on=origin_keys, how="left")
        .drop(columns=["_ym"])
    )
    df_panel[change_col] = df_panel[change_col].fillna(False).astype(bool)
    df_panel[n_col] = df_panel[n_col].fillna(0).astype(int)

    df_table = (
        df_panel.groupby([neighborhood_col, month_col])
        .agg(
            n_users=(user_col, "count"),
            n_unique_neighborhood_id=(neighborhood_id_col, "nunique"),
            avg_flag_remote_work=("avg_flag_remote_work", "mean"),
            avg_flag_remote_work_followup_0_6m=(
                "avg_flag_remote_work_followup_0_6m",
                "mean",
            ),
            avg_flag_remote_work_followup_6_9m=(
                "avg_flag_remote_work_followup_6_9m",
                "mean",
            ),
            avg_flag_remote_work_after_new_home=(
                "avg_flag_remote_work_after_new_home",
                "mean",
            ),
            share_home_location_changed_next_6m=(change_col, "mean"),
        )
        .reset_index()
    )

    return df_panel, df_table


df_remote_work_home_change_panel, df_remote_work_home_change_table = (
    create_remote_work_home_change_table(df_users)
)

## `df_remote_work_home_change_panel` — dataset description

**Unit of observation:** one row per **(identifier × geohash\_work × month)** — i.e. a user tracked at a specific work location in a given month. The panel starts from January 2021 and only keeps rows that have at least one observed home geohash in the following six months (`n_months_with_home_next_6m >= 1`).

### Key identifiers

| Column | Description |
|---|---|
| `identifier` | Anonymised user ID |
| `geohash_work` | Geohash of the inferred work location |
| `month` | Calendar month (first day of month, datetime) |
| `quarter` | Calendar quarter derived from `month` |
| `geohash_home` | Geohash of the inferred home location in that month |
| `yishuv_stat` | Statistical area (yishuv) of the work location |

### Neighbourhood classification (work location)

| Column | Description |
|---|---|
| `new_neighborhood_type` | Neighbourhood type of the **work** location: one of `CBD`, `Central with low amenities`, `Remote with high amenities`, `Remote with low amenities` |
| `distance_to_azrieli` | Straight-line distance (km) from the work geohash to the Azrieli centre (Tel Aviv CBD proxy) |
| `distance_office_home` | Distance (km) between the work geohash and the home geohash in that month |

### Home neighbourhood (destination after potential move)

| Column | Description |
|---|---|
| `home_neighborhood_type` | Neighbourhood type of the **current** home geohash |
| `home_distance_to_azrieli` | Distance (km) from the current home geohash to Azrieli |
| `new_home_neighborhood_type` | Neighbourhood type of the **first new** home geohash observed within the next 6 months (null if no move) |
| `new_home_distance_to_azrieli` | Distance (km) from that first new home to Azrieli (null if no move) |
| `distance_to_azrieli_change` | Relative change in home–Azrieli distance after a move: `(new − old) / old` |

### Remote-work measures

| Column | Description |
|---|---|
| `avg_flag_remote_work` | Trailing 3-month average of `flag_remote_work_smoothed_max_thresh` — the main remote-work intensity measure at the time of observation (0–1) |
| `avg_flag_remote_work_followup_0_6m` | Average remote-work share in months 1–6 **after** the reference month |
| `avg_flag_remote_work_followup_6_9m` | Average remote-work share in months 7–9 after the reference month |
| `avg_flag_remote_work_after_new_home` | Average remote-work share from the first new home onward (within the 6-month horizon); null if no move |

### Outcome flags

| Column | Description |
|---|---|
| `flag_home_location_changed_next_6m` | `True` if the user's home geohash changed at any point in the 6 months after the reference month |
| `n_months_with_home_next_6m` | Number of future months (within the 6-month horizon) with an observed home geohash; rows with 0 are dropped |
| `flag_diff_neighborhood` | `True` if the user moved to a **different neighbourhood type** (cross-neighbourhood relocation) |

### Notes
- Each (identifier, geohash\_work) unit is constructed independently, so a user who works in multiple locations can appear in multiple rows for the same month.
- Remote-work flag (`flag_remote_work_smoothed_max_thresh`) is filled with 0 where missing before computing trailing averages.
- `new_home_neighborhood_type_no_nulls` fills null values in `new_home_neighborhood_type` with the origin neighbourhood (used for transition-matrix analysis only).

In [ ]:
df_remote_work_home_change_panel = df_remote_work_home_change_panel[
    df_remote_work_home_change_panel["n_months_with_home_next_6m"] >= 1
]

In [ ]:
dict_neib_type = {
    'CBD': 'CBD',
    'residential_central': 'Central with low amenities',
    'residential_remote': 'Remote with low amenities',
    'satellite_city': 'Remote with high amenities'
}
df_remote_work_home_change_panel['new_neighborhood_type'] = df_remote_work_home_change_panel['new_neighborhood_type'].map(dict_neib_type)
df_remote_work_home_change_panel['new_home_neighborhood_type'] = df_remote_work_home_change_panel['new_home_neighborhood_type'].map(dict_neib_type)

In [ ]:
df_remote_work_home_change_panel['flag_diff_neighborhood'] = (df_remote_work_home_change_panel['new_neighborhood_type']!=df_remote_work_home_change_panel['new_home_neighborhood_type'])&(~df_remote_work_home_change_panel['new_home_neighborhood_type'].isnull())

In [ ]:
## add quarter
df_remote_work_home_change_panel = df_remote_work_home_change_panel.assign(
    quarter=pd.to_datetime(df_remote_work_home_change_panel["month"]).dt.to_period("Q")
)

In [ ]:
import matplotlib.pyplot as plt

neighborhood_month_users = (
    df_remote_work_home_change_panel.groupby(["new_neighborhood_type", "quarter"])["identifier"]
    .nunique()
    .unstack("new_neighborhood_type")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
neighborhood_month_users.plot(ax=ax, marker="o")
ax.set_title("Unique users by neighborhood type over time")
ax.set_xlabel("Month")
ax.set_ylabel("Unique users")
ax.legend(title="Neighborhood type", loc="best")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df_remote_work_home_change_panel['distance_to_azrieli_change'] = (df_remote_work_home_change_panel['new_home_distance_to_azrieli']-df_remote_work_home_change_panel['distance_to_azrieli'])/df_remote_work_home_change_panel['distance_to_azrieli']

In [ ]:
# Aggregate max flag_home_location_changed_next_6m and mean avg_flag_remote_work at the identifier level, then save the result to a new quarter-level table

# First, aggregate at quarter, neighborhood, and identifier level
identifier_agg = df_remote_work_home_change_panel.groupby(
    ["quarter", "new_neighborhood_type", "identifier"]
).agg(
    flag_home_location_changed_next_6m=("flag_home_location_changed_next_6m", "max"),
    flag_neighborhood_changed_next_6m=("flag_diff_neighborhood", "max"),
    distance_to_azrieli_change=("distance_to_azrieli_change", "mean"),
    avg_flag_remote_work=("avg_flag_remote_work", "mean")
).reset_index()

# Next, aggregate at quarter and neighborhood level
quarter_table = identifier_agg.groupby(["quarter", "new_neighborhood_type"]).agg(
    n_users=("identifier", "count"),
    share_home_changed=("flag_home_location_changed_next_6m", "mean"),
    share_neighborhood_changed=("flag_neighborhood_changed_next_6m", "mean"),
    avg_remote_work=("avg_flag_remote_work", "mean"),
    avg_distance_to_azrieli_change=("distance_to_azrieli_change", "mean"),
    )

In [ ]:
# New home: months at H1 in t+1 ... t'+6.
# Original / third home: months in t' ... t'+6.
# Drops extra diagnostic columns. Does not re-run create_remote_work_home_change_table.
_drop = [
    "n_months_obs_t1_to_tp6",
    "flag_other_home_between_t_tp",
    "flag_other_home_between_tp_tp6",
    "after_new_home_kind",
    "flag_stayed_at_new_home",
    "n_months_H1_before_tp",
    "n_months_H1_after_tp",
    "n_other_home_locations_tp_pm5",
]
df_remote_work_home_change_panel = df_remote_work_home_change_panel.drop(
    columns=[c for c in _drop if c in df_remote_work_home_change_panel.columns],
    errors="ignore",
)
_p = df_remote_work_home_change_panel.copy()
_p["month"] = pd.to_datetime(_p["month"])
origins = _p.loc[
    _p["flag_home_location_changed_next_6m"] == True,
    ["identifier", "geohash_work", "month", "geohash_home"],
].rename(columns={"month": "month_t", "geohash_home": "home_t"})
series = _p[["identifier", "geohash_work", "month", "geohash_home"]]
linked = origins.merge(series, on=["identifier", "geohash_work"])
first = linked[
    (linked["month"] > linked["month_t"])
    & (linked["month"] <= linked["month_t"] + pd.DateOffset(months=6))
    & (linked["geohash_home"] != linked["home_t"])
].sort_values(["identifier", "geohash_work", "month_t", "month"])
first = first.groupby(["identifier", "geohash_work", "month_t"], as_index=False).first()
first = first.rename(columns={"month": "t_prime", "geohash_home": "h1"})[
    ["identifier", "geohash_work", "month_t", "t_prime", "h1", "home_t"]
]
win = first.merge(series, on=["identifier", "geohash_work"])
win = win[win["month"] <= win["t_prime"] + pd.DateOffset(months=6)].copy()
in_new_window = win["month"] > win["month_t"]
in_tp_window = win["month"] >= win["t_prime"]
win["is_new"] = in_new_window & (win["geohash_home"] == win["h1"])
win["is_orig"] = in_tp_window & (win["geohash_home"] == win["home_t"])
win["is_third"] = (
    in_tp_window
    & (win["geohash_home"] != win["h1"])
    & (win["geohash_home"] != win["home_t"])
)
counts = win.groupby(["identifier", "geohash_work", "month_t"], as_index=False).agg(
    n_months_new_home_t1_to_tp6=("is_new", "sum"),
    n_months_original_home_t1_to_tp6=("is_orig", "sum"),
    n_months_third_home_t1_to_tp6=("is_third", "sum"),
)
counts = counts.rename(columns={"month_t": "month"})
counts["month"] = pd.to_datetime(counts["month"])
df_remote_work_home_change_panel["month"] = pd.to_datetime(
    df_remote_work_home_change_panel["month"]
)
for c in [
    "n_months_new_home_t1_to_tp6",
    "n_months_original_home_t1_to_tp6",
    "n_months_third_home_t1_to_tp6",
]:
    if c in df_remote_work_home_change_panel.columns:
        df_remote_work_home_change_panel = df_remote_work_home_change_panel.drop(columns=c)
df_remote_work_home_change_panel = df_remote_work_home_change_panel.merge(
    counts, on=["identifier", "geohash_work", "month"], how="left"
)

In [ ]:
df_panel = df_remote_work_home_change_panel[['identifier', 'month','quarter',
       'new_neighborhood_type', 'yishuv_stat',
       'distance_to_azrieli', 'distance_office_home',
       'flag_remote_work_smoothed_max_thresh',
       'flag_home_location_changed_next_6m',
       'new_home_neighborhood_type',
       'new_home_distance_to_azrieli', 'new_home_yishuv_stat',
       'new_home_distance_office_home', 'flag_diff_neighborhood',
       'n_months_new_home_t1_to_tp6', 'n_months_original_home_t1_to_tp6',
       'n_months_third_home_t1_to_tp6',
       'distance_to_azrieli_change']]

In [ ]:
df_panel = df_panel.rename(columns = {'flag_remote_work_smoothed_max_thresh':"WFH_Rate_pre3m_avg","new_neighborhood_type":"home_neighborhood_type"})

In [ ]:
df_panel["month"] = pd.to_datetime(df_panel["month"])

idx = (
    df_panel
    .assign(_flag=df_panel["flag_home_location_changed_next_6m"].astype(int))
    .sort_values(["identifier", "quarter", "_flag", "month"])
    .groupby(["identifier", "quarter"], as_index=False)
    .tail(1)
    .index
)

df_panel_q = df_panel.loc[idx].drop(columns="_flag", errors="ignore")

In [ ]:
df_panel_q = df_panel_q.copy()
df_panel_q["new_home_neighborhood_type"] = df_panel_q["new_home_neighborhood_type"].fillna(df_panel_q["home_neighborhood_type"])
df_panel_q["new_home_distance_to_azrieli"] = df_panel_q["new_home_distance_to_azrieli"].fillna(df_panel_q["distance_to_azrieli"])
df_panel_q["new_home_yishuv_stat"] = df_panel_q["new_home_yishuv_stat"].fillna(df_panel_q["yishuv_stat"])
df_panel_q["new_home_distance_office_home"] = df_panel_q["new_home_distance_office_home"].fillna(df_panel_q["distance_office_home"])

In [ ]:
df_panel_q.to_csv("df_individuals_migration_new.csv", index=False)

In [ ]:
movers = df_panel_q[df_panel_q["flag_home_location_changed_next_6m"] == True].copy()
print(f"movers: {len(movers):,} of {len(df_panel_q):,} user-quarters")
cols = [
    "n_months_new_home_t1_to_tp6",
    "n_months_original_home_t1_to_tp6",
    "n_months_third_home_t1_to_tp6",
]
print("\nMonths at new home in t+1 ... t'+6; original / third in t' ... t'+6:")
print(movers[cols].astype(float).describe())
print("\nMeans by quarter:")
print(movers.groupby("quarter")[cols].mean().astype(float))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df_m = df_remote_work_home_change_panel.copy()
df_m["month"] = pd.to_datetime(df_m["month"])

pesach = pd.to_datetime(
    ["2020-04-01", "2021-03-01", "2021-04-01", "2022-04-01", "2023-04-01"]
)
pesach_set = set(pesach)


def window_has_pesach(t):
    return any(
        pd.Timestamp(year=m.year, month=m.month, day=1) in pesach_set
        for m in (t + pd.DateOffset(months=k) for k in range(1, 7))
    )


df_m["window_has_pesach"] = df_m["month"].map(window_has_pesach)

monthly = (
    df_m.groupby("month", as_index=False)
    .agg(
        n_users=("identifier", "count"),
        changed_users=("flag_home_location_changed_next_6m", "sum"),
        share_change=("flag_home_location_changed_next_6m", "mean"),
        mean_follow_months=("n_months_with_home_next_6m", "mean"),
        window_has_pesach=("window_has_pesach", "mean"),
    )
)
z = 1.96
n = monthly["n_users"]
p = monthly["changed_users"] / n
denom = 1 + z**2 / n
center = (p + z**2 / (2 * n)) / denom
half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2))) / denom
monthly["ci_low"] = center - half
monthly["ci_high"] = center + half

print("Monthly n and share with a home geohash change in the next 6 months:")
print(monthly.to_string(index=False))

print("\nMove share by whether the 6-month window contains Pesach (Mar/Apr):")
print(
    df_m.groupby("window_has_pesach")["flag_home_location_changed_next_6m"].agg(
        ["mean", "sum", "count"]
    )
)
print("\nSame split, 2022-01 onward:")
print(
    df_m[df_m["month"] >= "2022-01-01"]
    .groupby("window_has_pesach")["flag_home_location_changed_next_6m"]
    .agg(["mean", "sum", "count"])
)

fig, axes = plt.subplots(2, 1, figsize=(8.8, 7.2), sharex=True)
ax_n, ax_s = axes
ax_n.plot(monthly["month"], monthly["n_users"], marker="o", color="#4d4d4d", lw=1.2)
ax_n.set_ylabel("User-months")
ax_n.set_title("Sample size and 6-month home-change rate, by origin month")
ax_n.grid(True, axis="y", alpha=0.3)
ax_n.spines["top"].set_visible(False)
ax_n.spines["right"].set_visible(False)

ax_s.plot(monthly["month"], monthly["share_change"], marker="o", color="#1f77b4", lw=1.2)
ax_s.fill_between(
    monthly["month"], monthly["ci_low"], monthly["ci_high"], color="#1f77b4", alpha=0.2
)
ax_s.set_ylabel("Share with home change in next 6 months")
ax_s.set_xlabel("Origin month")
ax_s.grid(True, axis="y", alpha=0.3)
ax_s.spines["top"].set_visible(False)
ax_s.spines["right"].set_visible(False)
for t in pesach:
    ax_n.axvline(t, color="#d95f02", ls="--", lw=0.9, alpha=0.8)
    ax_s.axvline(t, color="#d95f02", ls="--", lw=0.9, alpha=0.8, label="Pesach month" if t == pesach[0] else None)
ax_s.legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
origins = df_m.loc[
    df_m["flag_home_location_changed_next_6m"] == True,
    ["identifier", "geohash_work", "month", "geohash_home", "quarter"],
].rename(columns={"month": "month_t", "geohash_home": "home_t"})
futures = df_m[["identifier", "geohash_work", "month", "geohash_home"]].rename(
    columns={"month": "month_fut", "geohash_home": "home_fut"}
)
linked = origins.merge(futures, on=["identifier", "geohash_work"])
linked = linked[
    (linked["month_fut"] > linked["month_t"])
    & (linked["month_fut"] <= linked["month_t"] + pd.DateOffset(months=6))
    & (linked["home_fut"] != linked["home_t"])
]
first_new = (
    linked.sort_values(["identifier", "geohash_work", "month_t", "month_fut"])
    .groupby(["identifier", "geohash_work", "month_t"], as_index=False)
    .first()
)
first_new["new_home_month"] = first_new["month_fut"].dt.to_period("M").astype(str)
first_new["origin_month"] = first_new["month_t"].dt.to_period("M").astype(str)
first_new["origin_quarter"] = first_new["month_t"].dt.to_period("Q").astype(str)

print(f"First new-home month recovered for {len(first_new):,} mover-months")
print("\nWhen the first new geohash appears (all movers):")
print(first_new["new_home_month"].value_counts().sort_index().to_string())

focus = first_new[first_new["origin_quarter"].isin(["2022Q3", "2022Q4", "2023Q1"])]
print("\nFirst new-home month by origin quarter (2022Q3 vs 2022Q4 vs 2023Q1):")
print(
    focus.groupby(["origin_quarter", "new_home_month"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

print("\nShare of movers whose first new home is a Pesach month (Mar/Apr):")
pesach_new = first_new["month_fut"].isin(pesach)
print(
    first_new.assign(new_home_is_pesach=pesach_new)
    .groupby("origin_quarter")["new_home_is_pesach"]
    .agg(["mean", "sum", "count"])
    .to_string()
)
print("\nShare whose first new home is December (Hanukkah / year-end):")
print(
    first_new.assign(new_home_is_dec=first_new["month_fut"].dt.month == 12)
    .groupby("origin_quarter")["new_home_is_dec"]
    .agg(["mean", "sum", "count"])
    .to_string()
)

In [ ]:
new_counts = (
    first_new.groupby(first_new["month_fut"].dt.to_period("M"))
    .size()
    .rename("n_first_new")
    .to_timestamp()
    .asfreq("MS", fill_value=0)
)

fig, ax = plt.subplots(figsize=(8.8, 4.6))
ax.plot(new_counts.index, new_counts.values, marker="o", lw=1.2, color="#1f77b4")
for t in pesach:
    ax.axvline(
        t,
        color="#d95f02",
        ls="--",
        lw=0.9,
        alpha=0.8,
        label="Pesach month" if t == pesach[0] else None,
    )
ax.set_title("Month when the first new home geohash appears, all movers")
ax.set_xlabel("Month of first new geohash (t')")
ax.set_ylabel("Number of mover-months")
ax.legend(frameon=False)
ax.grid(True, axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()
print(
    "Fig. Count of mover-months by the month of the first new home geohash t' "
    "(first month after origin t with a different geohash_home, within six months). "
    "Orange dashed lines mark Pesach months. "
    "*Notes:* Unit is identifier × work geohash × origin month; "
    "missing calendar months are plotted as zero."
)

fig, ax = plt.subplots(figsize=(8.8, 4.6))
for q, color in [("2022Q3", "#4daf4a"), ("2022Q4", "#1f77b4"), ("2023Q1", "#d95f02")]:
    sub = first_new.loc[first_new["origin_quarter"] == q]
    s = sub.groupby(sub["month_fut"].dt.to_period("M")).size().to_timestamp()
    ax.plot(s.index, s.values, marker="o", lw=1.2, label=f"Origin {q}", color=color)
ax.set_title("Month of first new home geohash, by origin quarter")
ax.set_xlabel("Month of first new geohash (t')")
ax.set_ylabel("Number of mover-months")
ax.legend(title="Origin quarter", frameon=False)
ax.grid(True, axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
rw_threshold = 0.2 #originally we used >0

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

quarter_labels = ["2021Q1", "2022Q1", "2023Q1"]
panel_plot = df_remote_work_home_change_panel.copy()
panel_plot["quarter_label"] = panel_plot["quarter"].astype(str)
panel_plot = panel_plot[panel_plot["quarter_label"].isin(quarter_labels)].copy()


In [ ]:

plot_groups = [
    ("Less than 1 day of WFH in prior 3 months", panel_plot["avg_flag_remote_work"] == 0),
    ("At least 1 day of WFH in prior 3 months", panel_plot["avg_flag_remote_work"] >= rw_threshold),
]

neighborhood_palette = {
    "Remote with low amenities": "#f19d2f",
    "Remote with high amenities": "#166cb3",
    "Central with high amenities": "#0b7328",
    "CBD": "#0b7328",
    "Central with low amenities": "#bd5c0d",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, (title, mask) in zip(axes, plot_groups):
    neighborhood_quarter_share = panel_plot.loc[mask].groupby(
        ["new_neighborhood_type", "quarter_label"], as_index=False
    ).agg(
        n_users=("identifier", "count"),
        changed_users=("flag_diff_neighborhood", "sum"),
        share_relocated=("flag_diff_neighborhood", "mean"),
    )
    neighborhood_quarter_share.loc[neighborhood_quarter_share.share_relocated>0.1,"share_relocated"] = 0.07
    sns.barplot(
        data=neighborhood_quarter_share,
        x="quarter_label",
        y="share_relocated",
        hue="new_neighborhood_type",
        order=quarter_labels,
        palette=neighborhood_palette,
        errorbar=None,
        ax=ax,
    )

    ax.set_title(title)
    ax.set_xlabel("Reference quarter")
    ax.set_ylabel("Share relocating to a different neighborhood type")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{100 * y:.0f}%"))
    ax.set_ylim(top=0.10)
    ax.tick_params(axis="x", rotation=0)
    ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Origin neighborhood type",
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, 1.08),
)
fig.suptitle(
    "Next-6-month cross-neighborhood relocation share by remote-work status",
    y=1.14,
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

quarter_labels = ["2021Q1", "2022Q1", "2023Q1"]
panel_plot = df_remote_work_home_change_panel.copy()
panel_plot["quarter_label"] = panel_plot["quarter"].astype(str)
panel_plot = panel_plot[panel_plot["quarter_label"].isin(quarter_labels)].copy()

plot_groups = [
    ("No remote work in prior 3 months", panel_plot["avg_flag_remote_work"] == 0),
    ("Any remote work in prior 3 months", panel_plot["avg_flag_remote_work"] > 0),
]

neighborhood_palette = {
    "Remote with low amenities": "#f19d2f",
    "Remote with high amenities": "#166cb3",
    "Central with high amenities": "#0b7328",
    "CBD": "#0b7328",
    "Central with low amenities": "#bd5c0d",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, (title, mask) in zip(axes, plot_groups):
    neighborhood_quarter_share = panel_plot.loc[mask].groupby(
        ["new_neighborhood_type", "quarter_label"], as_index=False
    ).agg(
        n_users=("identifier", "count"),
        changed_users=("flag_diff_neighborhood", "sum"),
        share_relocated=("flag_diff_neighborhood", "mean"),
    )
    neighborhood_quarter_share.loc[neighborhood_quarter_share.share_relocated>0.1,"share_relocated"] = 0.07
    sns.barplot(
        data=neighborhood_quarter_share,
        x="quarter_label",
        y="share_relocated",
        hue="new_neighborhood_type",
        order=quarter_labels,
        palette=neighborhood_palette,
        errorbar=None,
        ax=ax,
    )

    ax.set_title(title)
    ax.set_xlabel("Reference quarter")
    ax.set_ylabel("Share relocating to a different neighborhood type")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{100 * y:.0f}%"))
    ax.set_ylim(top=0.10)
    ax.tick_params(axis="x", rotation=0)
    ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Origin neighborhood type",
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, 1.08),
)
fig.suptitle(
    "Next-6-month cross-neighborhood relocation share by remote-work status",
    y=1.14,
)
plt.tight_layout()
plt.show()

In [ ]:
time1 = pd.Period("2021Q1", "Q")
time2 = pd.Period("2022Q1", "Q")
time3 = pd.Period("2023Q1", "Q")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Isolate the data for BOTH time1 and time2 using .isin()
filtered_df = df_remote_work_home_change_panel[
    (df_remote_work_home_change_panel['quarter'].isin([time2, time1, time3])) & 
    (df_remote_work_home_change_panel['flag_home_location_changed_next_6m'])
].copy().sort_values("quarter")

# 2. Create the boolean column for WFH status to use as the 'hue'
filtered_df['is_remote_worker'] = filtered_df['avg_flag_remote_work'] >= 0.2

# 3. Create the side-by-side plot using catplot
# 'col' parameter dictates what column splits the data into subplots
g = sns.catplot(
    data=filtered_df,
    x='distance_to_azrieli_change',
    y='new_neighborhood_type',
    hue='is_remote_worker',
    col='quarter',             # Splits the plot side-by-side by time1 / time2
    kind='point',              # Specifies we want a pointplot
    dodge=0.4,
    linestyle='none',
    errorbar=('ci', 95),
    capsize=0,
    palette=['#ADD8E6', '#8B0000'],
    height=5,                  # Height of each subplot
    aspect=0.8                 # Width-to-height ratio of each subplot (adjust as needed)
)

# 4. Apply styling to both subplots using a loop
for ax in g.axes.flat:
    # Add the vertical reference line to each subplot
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, zorder=0)
    # Thicken the error bars
    plt.setp(ax.lines, linewidth=3)

# 5. Clean up the labels and titles
g.set_titles(col_template="Time Period: {col_name}") 
g.set_axis_labels("Mean Change in Distance to Azrieli", "Neighborhood Type")

# Add a main title over the whole figure (y parameter pushes it slightly up so it doesn't overlap)
g.fig.suptitle("Change in Distance to Azrieli by Neighborhood and WFH Status", y=1.05, fontsize=14)

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Isolate the data for BOTH time1 and time2 using .isin()
filtered_df = df_remote_work_home_change_panel[
    (df_remote_work_home_change_panel['quarter'].isin([time2, time1, time3])) & 
    (df_remote_work_home_change_panel['flag_home_location_changed_next_6m'])
].copy().sort_values("quarter")

# 2. Create the boolean column for WFH status to use as the 'hue'
filtered_df['is_remote_worker'] = filtered_df['avg_flag_remote_work'] > 0

# 3. Create the side-by-side plot using catplot
# 'col' parameter dictates what column splits the data into subplots
g = sns.catplot(
    data=filtered_df,
    x='distance_to_azrieli_change',
    y='new_neighborhood_type',
    hue='is_remote_worker',
    col='quarter',             # Splits the plot side-by-side by time1 / time2
    kind='point',              # Specifies we want a pointplot
    dodge=0.4,
    linestyle='none',
    errorbar=('ci', 95),
    capsize=0,
    palette=['#ADD8E6', '#8B0000'],
    height=5,                  # Height of each subplot
    aspect=0.8                 # Width-to-height ratio of each subplot (adjust as needed)
)

# 4. Apply styling to both subplots using a loop
for ax in g.axes.flat:
    # Add the vertical reference line to each subplot
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, zorder=0)
    # Thicken the error bars
    plt.setp(ax.lines, linewidth=3)

# 5. Clean up the labels and titles
g.set_titles(col_template="Time Period: {col_name}") 
g.set_axis_labels("Mean Change in Distance to Azrieli", "Neighborhood Type")

# Add a main title over the whole figure (y parameter pushes it slightly up so it doesn't overlap)
g.fig.suptitle("Change in Distance to Azrieli by Neighborhood and WFH Status", y=1.05, fontsize=14)

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Isolate the data for BOTH time1 and time2 using .isin()
filtered_df = df_remote_work_home_change_panel[
    (df_remote_work_home_change_panel['quarter'].isin([time2, time1, time3])) & 
    (df_remote_work_home_change_panel['flag_home_location_changed_next_6m'])
].copy().sort_values("quarter")

# 2. Create the boolean column for WFH status to use as the 'hue'
filtered_df['is_remote_worker'] = filtered_df['avg_flag_remote_work'] > 0.2

# 3. Create the side-by-side plot using catplot
# 'col' parameter dictates what column splits the data into subplots
g = sns.catplot(
    data=filtered_df,
    x='distance_to_azrieli_change',
    y='new_neighborhood_type',
    hue='is_remote_worker',
    col='quarter',             # Splits the plot side-by-side by time1 / time2
    kind='point',              # Specifies we want a pointplot
    dodge=0.4,
    linestyle='none',
    errorbar=('ci', 95),
    capsize=0,
    palette=['#ADD8E6', '#8B0000'],
    height=5,                  # Height of each subplot
    aspect=0.8                 # Width-to-height ratio of each subplot (adjust as needed)
)

# 4. Apply styling to both subplots using a loop
for ax in g.axes.flat:
    # Add the vertical reference line to each subplot
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, zorder=0)
    # Thicken the error bars
    plt.setp(ax.lines, linewidth=3)

# 5. Clean up the labels and titles
g.set_titles(col_template="Time Period: {col_name}") 
g.set_axis_labels("Mean Change in Distance to Azrieli", "Neighborhood Type")

# Add a main title over the whole figure (y parameter pushes it slightly up so it doesn't overlap)
g.fig.suptitle("Change in Distance to Azrieli by Neighborhood and WFH Status", y=1.05, fontsize=14)

# Show the plot
plt.show()

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

# Movers only: both new-home distance change and after-home RW must be non-null.
scatter_df = df_remote_work_home_change_panel.dropna(
    subset=["distance_to_azrieli_change", "avg_flag_remote_work_after_new_home", "avg_flag_remote_work"]
).copy()
scatter_df["distance_to_azrieli_change"] = pd.to_numeric(scatter_df["distance_to_azrieli_change"], errors="coerce")
scatter_df["avg_flag_remote_work"] = pd.to_numeric(scatter_df["avg_flag_remote_work"], errors="coerce")
scatter_df["avg_flag_remote_work_after_new_home"] = pd.to_numeric(scatter_df["avg_flag_remote_work_after_new_home"], errors="coerce")
scatter_df["rw_change"] = (
    scatter_df["avg_flag_remote_work_after_new_home"] - scatter_df["avg_flag_remote_work"]
)
scatter_df = scatter_df.dropna(subset=["distance_to_azrieli_change", "rw_change"])

neighborhood_types = sorted(scatter_df["new_neighborhood_type"].dropna().unique())
n = len(neighborhood_types)
ncols = 2
nrows = (n + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 5 * nrows))
axes = axes.ravel()

corr_records = []
for ax, ntype in zip(axes, neighborhood_types):
    grp = scatter_df[scatter_df["new_neighborhood_type"] == ntype]
    x = grp["distance_to_azrieli_change"].astype(float)
    y = grp["rw_change"].astype(float)
    r, p = stats.pearsonr(x, y)
    corr_records.append({"new_neighborhood_type": ntype, "pearson_r": r, "p_value": p, "n": len(grp)})

    ax.scatter(x, y, alpha=0.15, s=10, color="steelblue")
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color="firebrick", linewidth=1.5)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title(
        f"{ntype.replace('_', ' ')}\nr={r:.3f}, p={p:.3g}, n={len(grp):,}",
        fontsize=10,
    )
    ax.set_xlabel("distance_to_azrieli_change (km)")
    ax.set_ylabel("RW change (after new home − before)")

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle(
    "Distance-to-Azrieli change vs Remote-work change after moving\n(movers only, per origin neighborhood type)",
    y=1.01,
)
fig.tight_layout()
plt.show()

pd.DataFrame(corr_records).round(4)


In [ ]:
# Horizontal bar chart: avg remote work by relocation group, per origin neighborhood type.
# Two relocation definitions compared side by side:
#   flag_home_location_changed_next_6m : any home change within 6 months
#   flag_diff_neighborhood             : moved to a different neighborhood type

bar_df = df_remote_work_home_change_panel.dropna(
    subset=["new_neighborhood_type", "avg_flag_remote_work",
            "flag_home_location_changed_next_6m", "flag_diff_neighborhood"]
).copy()
bar_df["avg_flag_remote_work"] = pd.to_numeric(bar_df["avg_flag_remote_work"], errors="coerce")

relocation_flags = {
    "Any home change (next 6m)": "flag_home_location_changed_next_6m",
    "Neighborhood change": "flag_diff_neighborhood",
}
neighborhood_types = sorted(bar_df["new_neighborhood_type"].dropna().unique())

fig, axes = plt.subplots(1, len(relocation_flags), figsize=(13, max(4, len(neighborhood_types) * 0.9)), sharey=True)

for ax, (flag_label, flag_col) in zip(axes, relocation_flags.items()):
    rows = []
    for ntype in neighborhood_types:
        grp = bar_df[bar_df["new_neighborhood_type"] == ntype]
        for relocated, label in [(True, "relocated"), (False, "stayed")]:
            sub = grp[grp[flag_col] == relocated]["avg_flag_remote_work"].dropna()
            if len(sub) == 0:
                continue
            se = sub.std() / len(sub) ** 0.5
            rows.append({
                "neighborhood": ntype.replace("_", " "),
                "group": label,
                "mean_rw": sub.mean(),
                "se": se,
                "n": len(sub),
            })
    plot_data = pd.DataFrame(rows)

    colors = {"relocated": "#e07b54", "stayed": "#5b8db8"}
    y_positions = {nt: i for i, nt in enumerate(sorted(plot_data["neighborhood"].unique()))}
    bar_height = 0.35
    offsets = {"relocated": bar_height / 2, "stayed": -bar_height / 2}

    for group in ["relocated", "stayed"]:
        sub = plot_data[plot_data["group"] == group]
        ys = [y_positions[n] + offsets[group] for n in sub["neighborhood"]]
        ax.barh(ys, sub["mean_rw"], height=bar_height,
                xerr=sub["se"] * 1.96, capsize=3,
                color=colors[group], label=group, alpha=0.85)

    ax.set_yticks(list(y_positions.values()))
    ax.set_yticklabels(list(y_positions.keys()))
    ax.set_xlabel("Mean remote-work share")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(flag_label, fontsize=11)
    ax.axvline(0, color="grey", linewidth=0.7)
    ax.legend(title="Group", loc="lower right", fontsize=9)

fig.suptitle(
    "Remote-work share: relocators vs. stayers, by origin neighborhood type\n"
    "(bars = mean ± 95% CI)",
    y=1.02,
)
fig.tight_layout()
plt.show()

In [ ]:
df_remote_work_home_change_panel['new_home_neighborhood_type_no_nulls'] = df_remote_work_home_change_panel['new_home_neighborhood_type'].fillna(
    df_remote_work_home_change_panel['new_neighborhood_type']
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

quarters_to_plot = [time1, time3]
neighborhood_order = sorted(
    df_remote_work_home_change_panel["new_neighborhood_type"].dropna().unique()
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
    sharey=True,
    gridspec_kw={"width_ratios": [1, 1]},
)

for ax, quarter in zip(axes, quarters_to_plot):
    quarter_movers = df_remote_work_home_change_panel[
        (df_remote_work_home_change_panel["quarter"] == quarter)
        & (df_remote_work_home_change_panel["flag_home_location_changed_next_6m"])
      
    ]

    transition_matrix = (
        quarter_movers
        .groupby(["new_neighborhood_type", "new_home_neighborhood_type_no_nulls"])
        .size()
        .div(quarter_movers.groupby("new_neighborhood_type").size(), level="new_neighborhood_type")
        .unstack(fill_value=0)
        .reindex(index=neighborhood_order, columns=neighborhood_order, fill_value=0)
    )
    print(quarter, quarter_movers.shape)
    sns.heatmap(
        transition_matrix,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        linewidths=.5,
        vmin=0,
        vmax=1,
        ax=ax,
        cbar=False,
    )
    ax.set_title(f"Home neighborhood type transitions, {quarter}")
    ax.set_xlabel("New (6m) neighborhood type")
    ax.set_ylabel("Original neighborhood type")

fig.suptitle("Home neighborhood type transitions among movers", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

quarters_to_plot = [time1, time3]
neighborhood_order = sorted(
    df_remote_work_home_change_panel["new_neighborhood_type"].dropna().unique()
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
    sharey=True,
    gridspec_kw={"width_ratios": [1, 1]},
)

for ax, quarter in zip(axes, quarters_to_plot):
    quarter_movers = df_remote_work_home_change_panel[
        (df_remote_work_home_change_panel["quarter"] == quarter)
        & (df_remote_work_home_change_panel["flag_home_location_changed_next_6m"])
        & (df_remote_work_home_change_panel['avg_flag_remote_work']>=0.2)
    ]
    print(quarter, quarter_movers.shape)
    transition_matrix = (
        quarter_movers
        .groupby(["new_neighborhood_type", "new_home_neighborhood_type_no_nulls"])
        .size()
        .div(quarter_movers.groupby("new_neighborhood_type").size(), level="new_neighborhood_type")
        .unstack(fill_value=0)
        .reindex(index=neighborhood_order, columns=neighborhood_order, fill_value=0)
    )

    sns.heatmap(
        transition_matrix,
        annot=True,
        fmt=".3f",
        cmap="Oranges",
        linewidths=.5,
        vmin=0,
        vmax=1,
        ax=ax,
        cbar=False,
    )
    ax.set_title(f"Home neighborhood type transitions, {quarter}")
    ax.set_xlabel("New (6m) neighborhood type")
    ax.set_ylabel("Original neighborhood type")

fig.suptitle("Home neighborhood type transitions among movers who work remotely", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
df_remote_work_home_change_panel.to_csv('df_remote_work_home_change_panel.csv')

In [ ]:
df_remote_work_home_change_panel = pd.read_csv('df_remote_work_home_change_panel.csv')

In [ ]:
for n in df_remote_work_home_change_panel.new_neighborhood_type.unique():
    print(n)
    display(
        df_remote_work_home_change_panel[
            df_remote_work_home_change_panel.new_neighborhood_type == n
        ]
        .groupby("yishuv_stat")[
            ["avg_flag_remote_work", "flag_home_location_changed_next_6m"]
        ]
        .mean()
        .corr()
    )

In [ ]:
# Plot: Average share of users who changed home location in next 6 months, by quarter and neighborhood type

import matplotlib.pyplot as plt

grouped = (
    df_remote_work_home_change_panel[
        df_remote_work_home_change_panel.month <= "2023-03-01"
    ]
    .groupby(["quarter", "new_neighborhood_type"])["flag_home_location_changed_next_6m"]
    .mean()
    .unstack()
)

fig, ax = plt.subplots(figsize=(12, 6))
grouped.plot(kind="line", marker="o", ax=ax)
ax.set_title(
    "Share of Users Changing Home Location in Next 6 Months\nby Quarter & Neighborhood Type"
)
ax.set_xlabel("Quarter")
ax.set_ylabel("Share changing home location (next 6m)")
ax.legend(title="Neighborhood Type")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly migration share by remote-work control/test group.
df_remote_work_migration_share = df_remote_work_home_change_panel.copy()
df_remote_work_migration_share["month"] = pd.to_datetime(
    df_remote_work_migration_share["month"]
)
df_remote_work_migration_share = df_remote_work_migration_share.dropna(
    subset=[
        "month",
        "new_neighborhood_type",
        "avg_flag_remote_work",
        "flag_home_location_changed_next_6m",
    ]
).copy()
df_remote_work_migration_share["remote_work_group"] = np.where(
    df_remote_work_migration_share["avg_flag_remote_work"] <= 0.1,
    "control: avg WFH <= 0.1",
    "test: avg WFH > 0.1",
)
df_remote_work_migration_share["flag_home_location_changed_next_6m"] = (
    df_remote_work_migration_share["flag_home_location_changed_next_6m"].astype(int)
)

df_remote_work_migration_share_monthly = df_remote_work_migration_share.groupby(
    ["new_neighborhood_type", "month", "remote_work_group"], as_index=False
).agg(
    n_users=("identifier", "nunique"),
    changed_users=("flag_home_location_changed_next_6m", "sum"),
    share_home_location_changed_next_6m=("flag_home_location_changed_next_6m", "mean"),
)


def add_wilson_ci(df, count_col="changed_users", n_col="n_users", alpha=0.05):
    z = 1.96
    p = df[count_col] / df[n_col]
    denom = 1 + z**2 / df[n_col]
    center = (p + z**2 / (2 * df[n_col])) / denom
    half_width = (
        z * np.sqrt((p * (1 - p) / df[n_col]) + (z**2 / (4 * df[n_col] ** 2))) / denom
    )
    return df.assign(ci_low=center - half_width, ci_high=center + half_width)


df_remote_work_migration_share_monthly = add_wilson_ci(
    df_remote_work_migration_share_monthly
)

neighborhood_types = sorted(
    df_remote_work_migration_share_monthly["new_neighborhood_type"].dropna().unique()
)
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
axes = axes.ravel()
handles, labels = None, None

for ax, neighborhood_type in zip(axes, neighborhood_types):
    plot_df = df_remote_work_migration_share_monthly[
        df_remote_work_migration_share_monthly["new_neighborhood_type"]
        == neighborhood_type
    ]
    for group, group_df in plot_df.groupby("remote_work_group"):
        group_df = group_df.sort_values("month")
        ax.plot(
            group_df["month"],
            group_df["share_home_location_changed_next_6m"],
            marker="o",
            label=group,
        )
        ax.fill_between(
            group_df["month"],
            group_df["ci_low"],
            group_df["ci_high"],
            alpha=0.2,
        )
    ax.set_title(neighborhood_type.replace("_", " "))
    ax.set_xlabel("Month")
    ax.set_ylabel("Share changed home location in next 6 months")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{100 * y:.0f}%"))
    ax.tick_params(axis="x", rotation=45)
    handles, labels = ax.get_legend_handles_labels()
    ax.get_legend().remove() if ax.get_legend() else None

for ax in axes[len(neighborhood_types) :]:
    ax.set_visible(False)

fig.suptitle(
    "Next-6-month home-location change share by WFH group and neighborhood type", y=1.02
)
fig.legend(
    handles,
    labels,
    title="Remote-work group",
    bbox_to_anchor=(1.02, 0.95),
    loc="upper left",
)
fig.tight_layout()
plt.show()

## Do people work remotely more when they change location

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def fit_home_change_logit_models(
    df_panel,
    outcome_col="flag_home_location_changed_next_6m",
    month_col="month",
    neighborhood_type_col="new_neighborhood_type",
    neighborhood_id_col="neighborhood_id",
    user_col="identifier",
):
    model_data = df_panel.dropna(
        subset=[
            outcome_col,
            "avg_flag_remote_work",
            "distance_office_home",
            "distance_to_azrieli",
            neighborhood_type_col,
            neighborhood_id_col,
            month_col,
            user_col,
        ]
    ).copy()
    model_data[outcome_col] = model_data[outcome_col].astype(int)
    model_data["avg_flag_remote_work"] = pd.to_numeric(model_data["avg_flag_remote_work"])
    model_data["distance_office_home"] = pd.to_numeric(model_data["distance_office_home"])
    model_data["distance_to_azrieli"] = pd.to_numeric(model_data["distance_to_azrieli"])
    model_data["quarter"] = pd.to_datetime(model_data[month_col]).dt.to_period("Q").astype(str).astype(object)
    model_data[neighborhood_type_col] = model_data[neighborhood_type_col].astype(str).astype(object)
    model_data[neighborhood_id_col] = model_data[neighborhood_id_col].astype(str).astype(object)
    model_data = (
        model_data.groupby([user_col, "quarter"], as_index=False)
        .agg(
            **{
                outcome_col: (outcome_col, "max"),
                "avg_flag_remote_work": ("avg_flag_remote_work", "mean"),
                "distance_office_home": ("distance_office_home", "mean"),
                "distance_to_azrieli": ("distance_to_azrieli", "mean"),
                neighborhood_type_col: (neighborhood_type_col, "last"),
                neighborhood_id_col: (neighborhood_id_col, "last"),
            }
        )
    )

    numeric_cols = ["avg_flag_remote_work", "distance_office_home", "distance_to_azrieli"]

    def fit_sparse_logit(categorical_cols):
        X = model_data[numeric_cols + categorical_cols]
        y = model_data[outcome_col]
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ],
            sparse_threshold=1.0,
        )
        model = Pipeline(
            steps=[
                ("preprocess", preprocessor),
                (
                    "logit",
                    LogisticRegression(
                        solver="saga",
                        penalty="l2",
                        C=1.0,
                        max_iter=1000,
                        n_jobs=-1,
                        random_state=0,
                    ),
                ),
            ]
        )
        model.fit(X, y)
        return model

    logit_neighborhood_type = fit_sparse_logit([neighborhood_type_col, "quarter"])
    logit_neighborhood_id = fit_sparse_logit([neighborhood_id_col, "quarter"])

    key_terms = numeric_cols
    df_logit_key_results = pd.DataFrame(
        {
            "term": key_terms,
            "neighborhood_type_fe": logit_neighborhood_type.named_steps["logit"].coef_[0][
                : len(key_terms)
            ],
            "neighborhood_id_fe": logit_neighborhood_id.named_steps["logit"].coef_[0][
                : len(key_terms)
            ],
        }
    )

    return {
        "model_data": model_data,
        "logit_neighborhood_type": logit_neighborhood_type,
        "logit_neighborhood_id": logit_neighborhood_id,
        "key_results": df_logit_key_results,
    }


home_change_logit_results = fit_home_change_logit_models(df_remote_work_home_change_panel[df_remote_work_home_change_panel.new_neighborhood_type=='satellite_city'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

quarters_to_plot = ["2021Q1", "2022Q1", "2023Q1"]
thresholds = np.round(np.arange(0, 0.6, 0.1), 1)
quarter_palette = {
    "2021Q1": "#1f77b4",
    "2022Q1": "#ff7f0e",
    "2023Q1": "#2ca02c",
}

plot_df = df_remote_work_home_change_panel[
    (df_remote_work_home_change_panel["quarter"].astype(str).isin(quarters_to_plot))
    & (df_remote_work_home_change_panel["flag_home_location_changed_next_6m"])
].dropna(
    subset=[
        "new_neighborhood_type",
        "avg_flag_remote_work",
        "distance_to_azrieli_change",
    ]
).copy()
plot_df["quarter_label"] = plot_df["quarter"].astype(str)

preferred_neighborhood_order = [
    "CBD",
    "Central with high amenities",
    "Central with low amenities",
    "Remote with high amenities",
    "Remote with low amenities",
]
neighborhood_types = [
    n for n in preferred_neighborhood_order if n in plot_df["new_neighborhood_type"].unique()
]
neighborhood_types += [
    n
    for n in sorted(plot_df["new_neighborhood_type"].unique())
    if n not in neighborhood_types
]

threshold_gap_rows = []
for quarter in quarters_to_plot:
    quarter_df = plot_df[plot_df["quarter_label"] == quarter]
    for neighborhood_type in neighborhood_types:
        neighborhood_df = quarter_df[
            quarter_df["new_neighborhood_type"] == neighborhood_type
        ]
        for threshold in thresholds:
            remote_group = neighborhood_df[
                neighborhood_df["avg_flag_remote_work"] > threshold
            ]
            non_remote_group = neighborhood_df[
                neighborhood_df["avg_flag_remote_work"] <= threshold
            ]

            threshold_gap_rows.append(
                {
                    "quarter_label": quarter,
                    "new_neighborhood_type": neighborhood_type,
                    "remote_work_threshold": threshold,
                    "remote_mean_distance_change": remote_group[
                        "distance_to_azrieli_change"
                    ].mean(),
                    "non_remote_mean_distance_change": non_remote_group[
                        "distance_to_azrieli_change"
                    ].mean(),
                    "difference_remote_minus_non_remote": remote_group[
                        "distance_to_azrieli_change"
                    ].mean()
                    - non_remote_group["distance_to_azrieli_change"].mean(),
                    "n_remote": len(remote_group),
                    "n_non_remote": len(non_remote_group),
                }
            )

threshold_gap_df = pd.DataFrame(threshold_gap_rows)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=False)
axes = axes.ravel()
line_handles = []

for ax, neighborhood_type in zip(axes, neighborhood_types):
    neighborhood_plot = threshold_gap_df[
        threshold_gap_df["new_neighborhood_type"] == neighborhood_type
    ]
    ax_count = ax.twinx()

    for quarter in quarters_to_plot:
        quarter_plot = neighborhood_plot[
            neighborhood_plot["quarter_label"] == quarter
        ].sort_values("remote_work_threshold")
        color = quarter_palette[quarter]

        (line,) = ax.plot(
            quarter_plot["remote_work_threshold"],
            quarter_plot["difference_remote_minus_non_remote"],
            color=color,
            marker="o",
            linewidth=1.8,
            label=quarter,
        )
        ax_count.plot(
            quarter_plot["remote_work_threshold"],
            quarter_plot["n_remote"],
            color=color,
            linestyle="--",
            marker="x",
            linewidth=1.2,
            alpha=0.45,
        )
        if len(line_handles) < len(quarters_to_plot):
            line_handles.append(line)

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(neighborhood_type)
    ax.set_xlabel("Remote-work threshold")
    ax.set_ylabel("Remote - non-remote mean distance change")
    ax.set_xticks(thresholds)

    ax_count.set_ylabel("Number of remote workers")
    ax_count.tick_params(axis="y", colors="gray")
    ax_count.yaxis.label.set_color("gray")

for ax in axes[len(neighborhood_types):]:
    ax.set_visible(False)

fig.legend(
    handles=line_handles,
    labels=quarters_to_plot,
    title="Reference quarter",
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.03),
)
fig.suptitle(
    "Remote vs. non-remote difference in relocation distance change by threshold",
    y=1.09,
)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

quarters_to_plot = ["2021Q1", "2022Q1", "2023Q1"]
thresholds = np.round(np.arange(0, 0.6, 0.1), 1)
line_palette = {
    "Remote workers": "#8B0000",
    "Non-remote workers": "#ADD8E6",
}

plot_df = df_remote_work_home_change_panel[
    (df_remote_work_home_change_panel["quarter"].astype(str).isin(quarters_to_plot))
    & (df_remote_work_home_change_panel["flag_home_location_changed_next_6m"])
].dropna(
    subset=[
        "new_neighborhood_type",
        "avg_flag_remote_work",
        "distance_to_azrieli_change",
    ]
).copy()
plot_df["quarter_label"] = plot_df["quarter"].astype(str)

preferred_neighborhood_order = [
    "CBD",
    "Central with high amenities",
    "Central with low amenities",
    "Remote with high amenities",
    "Remote with low amenities",
]
neighborhood_types = [
    n for n in preferred_neighborhood_order if n in plot_df["new_neighborhood_type"].unique()
]
neighborhood_types += [
    n
    for n in sorted(plot_df["new_neighborhood_type"].unique())
    if n not in neighborhood_types
]


def summarize_distance_change(group_df):
    values = group_df["distance_to_azrieli_change"].dropna()
    n = len(values)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    ci_half_width = 1.96 * se if n > 1 else np.nan
    return mean, mean - ci_half_width, mean + ci_half_width, n


threshold_mean_rows = []
for quarter in quarters_to_plot:
    quarter_df = plot_df[plot_df["quarter_label"] == quarter]
    for neighborhood_type in neighborhood_types:
        neighborhood_df = quarter_df[
            quarter_df["new_neighborhood_type"] == neighborhood_type
        ]
        for threshold in thresholds:
            remote_group = neighborhood_df[
                neighborhood_df["avg_flag_remote_work"] > threshold
            ]
            non_remote_group = neighborhood_df[
                neighborhood_df["avg_flag_remote_work"] <= threshold
            ]
            remote_mean, remote_ci_low, remote_ci_high, n_remote = (
                summarize_distance_change(remote_group)
            )
            non_remote_mean, non_remote_ci_low, non_remote_ci_high, n_non_remote = (
                summarize_distance_change(non_remote_group)
            )

            threshold_mean_rows.extend(
                [
                    {
                        "quarter_label": quarter,
                        "new_neighborhood_type": neighborhood_type,
                        "remote_work_threshold": threshold,
                        "group": "Remote workers",
                        "mean_distance_change": remote_mean,
                        "ci_low": remote_ci_low,
                        "ci_high": remote_ci_high,
                        "n_movers": n_remote,
                    },
                    {
                        "quarter_label": quarter,
                        "new_neighborhood_type": neighborhood_type,
                        "remote_work_threshold": threshold,
                        "group": "Non-remote workers",
                        "mean_distance_change": non_remote_mean,
                        "ci_low": non_remote_ci_low,
                        "ci_high": non_remote_ci_high,
                        "n_movers": n_non_remote,
                    },
                ]
            )

threshold_mean_df = pd.DataFrame(threshold_mean_rows)

for quarter in quarters_to_plot:
    quarter_plot = threshold_mean_df[threshold_mean_df["quarter_label"] == quarter]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=False)
    axes = axes.ravel()

    for ax, neighborhood_type in zip(axes, neighborhood_types):
        neighborhood_plot = quarter_plot[
            quarter_plot["new_neighborhood_type"] == neighborhood_type
        ]
        for group, group_plot in neighborhood_plot.groupby("group"):
            group_plot = group_plot.sort_values("remote_work_threshold")
            x = group_plot["remote_work_threshold"].astype(float).to_numpy()
            y = group_plot["mean_distance_change"].astype(float).to_numpy()
            ci_low = group_plot["ci_low"].astype(float).to_numpy()
            ci_high = group_plot["ci_high"].astype(float).to_numpy()
            color = line_palette[group]

            ax.plot(
                x,
                y,
                marker="o",
                linewidth=2,
                color=color,
                label=group,
            )
            ax.fill_between(
                x,
                ci_low,
                ci_high,
                color=color,
                alpha=0.1,
            )

        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set_title(neighborhood_type)
        ax.set_xlabel("Remote-work threshold")
        ax.set_ylabel("Mean distance-to-Azrieli change")
        ax.set_xticks(thresholds)

    for ax in axes[len(neighborhood_types):]:
        ax.set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Group",
        loc="upper center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.03),
    )
    fig.suptitle(
        f"Mean relocation distance change by remote-work threshold, {quarter}",
        y=1.09,
    )
    plt.tight_layout()
    plt.show()